In [1]:
import torch
?torch.tensor

Docstring:
tensor(data, *, dtype=None, device=None, requires_grad=False, pin_memory=False) -> Tensor

Constructs a tensor with no autograd history (also known as a "leaf tensor", see :doc:`/notes/autograd`) by copying :attr:`data`.

.. warning::

    When working with tensors prefer using :func:`torch.Tensor.clone`,
    :func:`torch.Tensor.detach`, and :func:`torch.Tensor.requires_grad_` for
    readability. Letting `t` be a tensor, ``torch.tensor(t)`` is equivalent to
    ``t.detach().clone()``, and ``torch.tensor(t, requires_grad=True)``
    is equivalent to ``t.detach().clone().requires_grad_(True)``.

.. seealso::

    :func:`torch.as_tensor` preserves autograd history and avoids copies where possible.
    :func:`torch.from_numpy` creates a tensor that shares storage with a NumPy array.

Args:
    data (array_like): Initial data for the tensor. Can be a list, tuple,
        NumPy ``ndarray``, scalar, and other types.

Keyword args:
    dtype (:class:`torch.dtype`, optional): the de

why prefer using torch.Tensor.clone? 使用者这样用的多吗？

让我从几个角度解释为什么推荐用 .clone() 以及实际使用情况。

为什么推荐用 .clone()
1. 语义明确性


# 情况1：从列表创建张量
x = torch.tensor([1, 2, 3])  # 清楚：创建新张量

# 情况2：从已有张量创建
t = torch.randn(3, 3)
x = torch.tensor(t)          # 不清楚：看起来像创建，实际是复制
x = t.clone()                # 清楚：就是复制
torch.tensor() 的设计初衷是从非张量数据创建张量，当输入已经是张量时，用 .clone() 能让读代码的人立刻明白"这是在复制一个已有张量"。

2. 梯度处理的可见性


t = torch.randn(3, requires_grad=True)

# 写法1：隐藏了 detach 操作
x = torch.tensor(t)  # 等价于 t.detach().clone()，但看不出来

# 写法2：明确显示切断了梯度
x = t.detach().clone()  # 一眼就看出来这个张量不会参与梯度计算
实际使用情况
真实代码中的使用习惯：

从列表/numpy 创建张量时，所有人都用：


x = torch.tensor([1, 2, 3])
x = torch.tensor(numpy_array)
复制已有张量时，分两种情况：

情况A - 需要保留梯度图： 几乎所有人用 .clone()


# 训练代码中常见
loss = model(x)
loss_copy = loss.clone()  # 保留梯度连接
情况B - 不需要梯度： 混合使用


# 官方教程和库代码更倾向于
x_copy = t.detach().clone()

# 但实际项目中也常看到
x_copy = torch.tensor(t.detach().cpu().numpy())  # 尤其是需要转到CPU时
所以实际情况是：

经验丰富的开发者和官方代码：倾向用 .clone() / .detach().clone()
初学者和快速原型代码：两种都用，torch.tensor(t) 也很常见
这个"警告"更多是代码规范建议，不是强制要求，两种写法功能完全等价
总结： 这不是性能问题，纯粹是代码可读性和语义清晰度的考虑。.clone() 让意图更明确，但 torch.tensor(t) 也不算错误写法，只是不够直观。